In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Fraud Detection Data Exploration\n",
    "## East African Rental Market\n",
    "**Participant:** Frank Karani | **Challenge:** #04 | **Country:** Tanzania\n",
    "\n",
    "This notebook explores the fraud detection methodology adapted from credit card fraud detection to rental listings."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Import Libraries"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "print(\"Libraries imported successfully\")\n",
    "print(f\"Pandas version: {pd.__version__}\")\n",
    "print(f\"NumPy version: {np.__version__}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. East African Market Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# East African rental market prices (USD per month)\n",
    "market_prices = {\n",
    "    'City': ['Dar es Salaam', 'Nairobi', 'Kampala', 'Arusha', 'Mombasa', 'Zanzibar'],\n",
    "    'Country': ['Tanzania', 'Kenya', 'Uganda', 'Tanzania', 'Kenya', 'Tanzania'],\n",
    "    '1BR': [350, 400, 300, 250, 300, 400],\n",
    "    '2BR': [550, 650, 500, 400, 500, 650],\n",
    "    '3BR': [800, 950, 750, 600, 700, 1000],\n",
    "    '4BR': [1200, 1400, 1100, 900, 1000, 1500]\n",
    "}\n",
    "\n",
    "df_market = pd.DataFrame(market_prices)\n",
    "df_market"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize market prices\n",
    "fig, ax = plt.subplots(figsize=(10, 6))\n",
    "df_market.set_index('City')[['1BR', '2BR', '3BR', '4BR']].plot(kind='bar', ax=ax)\n",
    "ax.set_title('East African Rental Market Prices (USD/month)', fontsize=14)\n",
    "ax.set_xlabel('City')\n",
    "ax.set_ylabel('Price (USD)')\n",
    "ax.legend(title='Bedrooms')\n",
    "plt.xticks(rotation=45)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Methodology Transfer: Credit Card Fraud → Rental Fraud"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Mapping credit card fraud features to rental fraud indicators\n",
    "mapping = {\n",
    "    'Credit Card Feature': ['V1 (PCA)', 'V2 (PCA)', 'V3 (PCA)', 'V4 (PCA)', 'V5 (PCA)', 'Time', 'Amount', 'Class'],\n",
    "    'Rental Feature': [\n",
    "        'Description quality score',\n",
    "        'Price anomaly vs market',\n",
    "        'User registration pattern',\n",
    "        'Image quality detection',\n",
    "        'Response time patterns',\n",
    "        'Listing frequency (rapid listing)',\n",
    "        'Rental price',\n",
    "        'Fraud label'\n",
    "    ],\n",
    "    'Weight in Final Score': ['15%', '35%', '10%', '10%', '5%', '15%', '10%', '-']\n",
    "}\n",
    "\n",
    "df_mapping = pd.DataFrame(mapping)\n",
    "df_mapping"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Fraud Detection Factors"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Factor weights for fraud detection\n",
    "factors = {\n",
    "    'Factor': ['Price Anomaly', 'User Behavior', 'Content Analysis', 'Geographic', 'Metadata'],\n",
    "    'Weight': [35, 25, 20, 10, 10],\n",
    "    'Description': [\n",
    "        'Compares price against East African market data',\n",
    "        'Account age, verification status',\n",
    "        'Keywords, images, description quality',\n",
    "        'City and location validation',\n",
    "        'Listing ID, price patterns'\n",
    "    ]\n",
    "}\n",
    "\n",
    "df_factors = pd.DataFrame(factors)\n",
    "df_factors"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize factor weights\n",
    "fig, ax = plt.subplots(figsize=(8, 6))\n",
    "colors = ['#ff6b6b', '#feca57', '#48dbfb', '#1dd1a1', '#5f27cd']\n",
    "ax.pie(df_factors['Weight'], labels=df_factors['Factor'], autopct='%1.0f%%', colors=colors, startangle=90)\n",
    "ax.set_title('Fraud Detection Factor Weights', fontsize=14)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Sample Data Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load sample data\n",
    "import os\n",
    "sample_path = '../app/data/sample_data.csv'\n",
    "if os.path.exists(sample_path):\n",
    "    df_samples = pd.read_csv(sample_path)\n",
    "    print(\"Sample data loaded successfully\")\n",
    "else:\n",
    "    print(f\"File not found: {sample_path}\")\n",
    "    df_samples = None"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "if df_samples is not None:\n",
    "    df_samples.head(10)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "if df_samples is not None:\n",
    "    fraud_counts = df_samples['is_fraud'].value_counts()\n",
    "    print(f\"Legitimate listings: {fraud_counts.get(False, 0)}\")\n",
    "    print(f\"Fraudulent listings: {fraud_counts.get(True, 0)}\")\n",
    "    print(f\"Fraud rate: {fraud_counts.get(True, 0) / len(df_samples) * 100:.1f}%\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "if df_samples is not None:\n",
    "    fig, ax = plt.subplots(figsize=(8, 5))\n",
    "    df_samples.boxplot(column='price', by='is_fraud', ax=ax)\n",
    "    ax.set_title('Price Distribution by Fraud Status')\n",
    "    ax.set_xlabel('Is Fraud')\n",
    "    ax.set_ylabel('Price (USD)')\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Suspicious Keywords Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "suspicious_keywords = {\n",
    "    'Keyword': ['urgent', 'deposit', 'western union', 'moneygram', 'no viewing', 'haraka', 'amana'],\n",
    "    'Language': ['English', 'English', 'English', 'English', 'English', 'Swahili', 'Swahili'],\n",
    "    'Risk Weight': [15, 20, 30, 30, 40, 15, 20]\n",
    "}\n",
    "\n",
    "df_keywords = pd.DataFrame(suspicious_keywords)\n",
    "df_keywords"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize keyword risk weights\n",
    "fig, ax = plt.subplots(figsize=(10, 5))\n",
    "colors_kw = ['#dc2626' if w >= 25 else '#f59e0b' for w in df_keywords['Risk Weight']]\n",
    "ax.barh(df_keywords['Keyword'], df_keywords['Risk Weight'], color=colors_kw)\n",
    "ax.set_xlabel('Risk Weight')\n",
    "ax.set_title('Suspicious Keywords Risk Weights')\n",
    "ax.invert_yaxis()\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Summary of Findings"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\" * 50)\n",
    "print(\"FRAUD DETECTION SUMMARY\")\n",
    "print(\"=\" * 50)\n",
    "print(f\"\\n1. East African Cities Supported: {len(df_market)}\")\n",
    "print(f\"2. Fraud Detection Factors: {len(df_factors)}\")\n",
    "print(f\"3. Suspicious Keywords Tracked: {len(df_keywords)}\")\n",
    "print(f\"4. Primary Fraud Indicator: Price Anomaly (35% weight)\")\n",
    "print(f\"5. Secondary Indicator: User Behavior (25% weight)\")\n",
    "print(\"\\n\" + \"=\" * 50)\n",
    "print(\"Methodology successfully transferred from\")\n",
    "print(\"Credit Card Fraud → Rental Listing Fraud\")\n",
    "print(\"=\" * 50)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.11.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}